In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q sentence-transformers transformers gensim scikit-learn

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 76.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import os
import time

# Đường dẫn thư mục (Tùy chỉnh theo Drive của nhóm)
BASE_PATH = "/content/drive/MyDrive/ĐATN/data"
INPUT_CSV = f"{BASE_PATH}/processed/metadata/df_multimodal_prompts.csv"
OUTPUT_DIR = f"{BASE_PATH}/processed/embeddings/text_features"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Đọc dữ liệu
print(f"Đang đọc dữ liệu từ: {INPUT_CSV}")
df = pd.read_csv(INPUT_CSV)

# Lấy danh sách ID sản phẩm và text tương ứng theo file CSV mới
image_ids = df['image_id'].values
item_ids = df['item_id'].values  # <--- Đã sửa 'product_id' thành 'item_id'
text_only_inputs = df['text_only_input'].fillna("").values
fashion_clip_prompts = df['fashion_clip_prompt'].fillna("").values

# Lưu mảng ID để sau này map với vector
np.save(f"{OUTPUT_DIR}/image_ids.npy", image_ids)
np.save(f"{OUTPUT_DIR}/item_ids.npy", item_ids)  # <--- Đã sửa tên file lưu thành 'item_ids.npy'
print(f"Tổng số mẫu văn bản: {len(df)}")

Đang đọc dữ liệu từ: /content/drive/MyDrive/ĐATN/data/processed/metadata/df_multimodal_prompts.csv
Tổng số mẫu văn bản: 12694


TF-IDF Extraction

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("--- 1. Đang trích xuất với TF-IDF ---")
start_time = time.time()

# Khởi tạo mô hình (giới hạn 5000 chiều để tránh mảng quá lớn)
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

# Huấn luyện và trích xuất dựa trên mô tả gốc (text_only_input)
tfidf_vectors = tfidf_vectorizer.fit_transform(text_only_inputs).toarray()

# Lưu file .npy
tfidf_path = f"{OUTPUT_DIR}/tfidf_embeddings.npy"
np.save(tfidf_path, tfidf_vectors)

print(f"Hoàn tất TF-IDF: {tfidf_vectors.shape}")
print(f"Thời gian: {time.time() - start_time:.2f}s")
print(f"Đã lưu tại: {tfidf_path}\n")

--- 1. Đang trích xuất với TF-IDF ---
Hoàn tất TF-IDF: (12694, 100)
Thời gian: 0.44s
Đã lưu tại: /content/drive/MyDrive/ĐATN/data/processed/embeddings/text_features/tfidf_embeddings.npy



FastText Extraction

In [ ]:
from gensim.models import FastText
import numpy as np

print("--- 2. Đang trích xuất với FastText ---")
start_time = time.time()

# Tokenize các câu thành list of words
tokenized_sentences = [text.split() for text in text_only_inputs]

# Huấn luyện mô hình FastText (chiều vector = 300)
ft_model = FastText(sentences=tokenized_sentences, vector_size=300, window=5, min_count=1, workers=4)

# Hàm chuyển một câu thành 1 vector duy nhất bằng cách trung bình cộng các từ
def get_sentence_vector(model, words):
    valid_words = [word for word in words if word in model.wv]
    if not valid_words:
        return np.zeros(model.vector_size)
    return np.mean([model.wv[word] for word in valid_words], axis=0)

fasttext_vectors = np.array([get_sentence_vector(ft_model, words) for words in tokenized_sentences])

# Lưu file .npy
ft_path = f"{OUTPUT_DIR}/fasttext_embeddings.npy"
np.save(ft_path, fasttext_vectors)

print(f"Hoàn tất FastText: {fasttext_vectors.shape}")
print(f"Thời gian: {time.time() - start_time:.2f}s")
print(f"Đã lưu tại: {ft_path}\n")

--- 2. Đang trích xuất với FastText ---
Hoàn tất FastText: (12694, 300)
Thời gian: 17.35s
Đã lưu tại: /content/drive/MyDrive/ĐATN/data/processed/embeddings/text_features/fasttext_embeddings.npy



SBERT Extraction

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

print("--- 3. Đang trích xuất với SBERT ---")
start_time = time.time()

# Thiết lập GPU nếu có
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Đang sử dụng thiết bị: {device}")

# Tải mô hình SBERT nhẹ và phổ biến nhất
sbert_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# Trích xuất vector
# batch_size=64 giúp xử lý nhanh hơn trên GPU
sbert_vectors = sbert_model.encode(text_only_inputs, batch_size=64, show_progress_bar=True)

# Lưu file .npy
sbert_path = f"{OUTPUT_DIR}/sbert_embeddings.npy"
np.save(sbert_path, sbert_vectors)

print(f"Hoàn tất SBERT: {sbert_vectors.shape}")
print(f"Thời gian: {time.time() - start_time:.2f}s")
print(f"Đã lưu tại: {sbert_path}\n")

--- 3. Đang trích xuất với SBERT ---
Đang sử dụng thiết bị: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/199 [00:00<?, ?it/s]

Hoàn tất SBERT: (12694, 384)
Thời gian: 16.15s
Đã lưu tại: /content/drive/MyDrive/ĐATN/data/processed/embeddings/text_features/sbert_embeddings.npy



Fashion-CLIP Extraction

In [ ]:
from transformers import CLIPTokenizer, CLIPTextModel
import torch

print("--- 4. Đang trích xuất với Fashion-CLIP (Text Branch) ---")
start_time = time.time()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = "patrickjohncyh/fashion-clip"

# Tải Tokenizer và mô hình Text của CLIP
tokenizer = CLIPTokenizer.from_pretrained(model_name)
text_model = CLIPTextModel.from_pretrained(model_name).to(device)
text_model.eval() # Chế độ đánh giá, không cập nhật gradient

BATCH_SIZE = 64
clip_vectors = []

# Trích xuất theo từng batch để không bị tràn RAM GPU (OOM)
with torch.no_grad():
    for i in range(0, len(fashion_clip_prompts), BATCH_SIZE):
        batch_texts = fashion_clip_prompts[i:i + BATCH_SIZE].tolist()

        # Tiền xử lý text
        inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt", max_length=77).to(device)

        # Đưa qua mô hình
        outputs = text_model(**inputs)

        # Lấy vector đại diện cho cả câu (pooler_output)
        embeddings = outputs.pooler_output.cpu().numpy()
        clip_vectors.append(embeddings)

        if (i % 1000 == 0) and i > 0:
            print(f"Đã xử lý {i} / {len(fashion_clip_prompts)} mẫu...")

# Gộp các batch lại thành mảng numpy duy nhất
clip_vectors_np = np.vstack(clip_vectors)

# Lưu file .npy
clip_path = f"{OUTPUT_DIR}/fashionclip_text_embeddings.npy"
np.save(clip_path, clip_vectors_np)

print(f"Hoàn tất Fashion-CLIP Text: {clip_vectors_np.shape}")
print(f"Thời gian: {time.time() - start_time:.2f}s")
print(f"Đã lưu tại: {clip_path}\n")

--- 4. Đang trích xuất với Fashion-CLIP (Text Branch) ---


tokenizer_config.json:   0%|          | 0.00/568 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] CLIPTextModel LOAD REPORT from: patrickjohncyh/fashion-clip
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.pre_layrnorm.bias                                 | UNEXPECTED |  | 
vision_model.encoder.la

Đã xử lý 8000 / 12694 mẫu...
Hoàn tất Fashion-CLIP Text: (12694, 512)
Thời gian: 38.80s
Đã lưu tại: /content/drive/MyDrive/ĐATN/data/processed/embeddings/text_features/fashionclip_text_embeddings.npy



In [ ]:
import numpy as np

print("--- KIỂM TRA KẾT QUẢ TRÍCH XUẤT VECTOR (SANITY CHECK) ---")

# 1. Lấy thông tin văn bản của mẫu ở vị trí index = 0
sample_idx = 0
sample_image_id = image_ids[sample_idx]
sample_text = text_only_inputs[sample_idx]
sample_clip_text = fashion_clip_prompts[sample_idx]

print(f"Ảnh ID: {sample_image_id}")
print(f"Text gốc (SBERT, TF-IDF, FastText): {sample_text}")
print(f"Text Prompt (Fashion-CLIP): {sample_clip_text}\n")

# 2. Đọc lại các vector vừa lưu từ ổ cứng
try:
    tfidf_vec = np.load(f"{OUTPUT_DIR}/tfidf_embeddings.npy")[sample_idx]
    ft_vec = np.load(f"{OUTPUT_DIR}/fasttext_embeddings.npy")[sample_idx]
    sbert_vec = np.load(f"{OUTPUT_DIR}/sbert_embeddings.npy")[sample_idx]
    clip_vec = np.load(f"{OUTPUT_DIR}/fashionclip_text_embeddings.npy")[sample_idx]

    print("1. TF-IDF Vector (Đặc trưng rời rạc - Sparse):")
    print(f"   - Số chiều (Shape): {tfidf_vec.shape}")
    print(f"   - Trích xuất 5 giá trị đầu: {tfidf_vec[:5]}")
    # TF-IDF thường có rất nhiều số 0 vì nó chỉ đếm từ vựng có trong câu
    print(f"   - Số lượng phần tử khác 0: {np.count_nonzero(tfidf_vec)} / {len(tfidf_vec)}\n")

    print("2. FastText Vector (Đặc trưng dày đặc - Dense):")
    print(f"   - Số chiều (Shape): {ft_vec.shape}")
    print(f"   - Trích xuất 5 giá trị đầu: {ft_vec[:5]}\n")

    print("3. SBERT Vector (Đặc trưng ngữ nghĩa - Dense):")
    print(f"   - Số chiều (Shape): {sbert_vec.shape}")
    print(f"   - Trích xuất 5 giá trị đầu: {sbert_vec[:5]}\n")

    print("4. Fashion-CLIP Vector (Đặc trưng đa phương thức - Dense):")
    print(f"   - Số chiều (Shape): {clip_vec.shape}")
    print(f"   - Trích xuất 5 giá trị đầu: {clip_vec[:5]}\n")

    print("KẾT LUẬN: Tất cả các file .npy đã lưu trữ dữ liệu hợp lệ (không trống).")

except FileNotFoundError as e:
    print(f"Lỗi: Không tìm thấy file vector. Hãy đảm bảo bạn đã chạy xong 4 Cell trước đó. Chi tiết: {e}")

--- KIỂM TRA KẾT QUẢ TRÍCH XUẤT VECTOR (SANITY CHECK) ---
Ảnh ID: MEN-Denim-id_00000080-01_7_additional
Text gốc (SBERT, TF-IDF, FastText): the lower clothing is of long length the fabric is cotton and it has plaid patterns
Text Prompt (Fashion-CLIP): a photo of a men denim, the lower clothing is of long length the fabric is cotton and it has plaid patterns

1. TF-IDF Vector (Đặc trưng rời rạc - Sparse):
   - Số chiều (Shape): (100,)
   - Trích xuất 5 giá trị đầu: [0.         0.         0.         0.12305056 0.        ]
   - Số lượng phần tử khác 0: 14 / 100

2. FastText Vector (Đặc trưng dày đặc - Dense):
   - Số chiều (Shape): (300,)
   - Trích xuất 5 giá trị đầu: [ 0.18377145  0.05024484  0.29306546  0.63610744 -0.01391644]

3. SBERT Vector (Đặc trưng ngữ nghĩa - Dense):
   - Số chiều (Shape): (384,)
   - Trích xuất 5 giá trị đầu: [ 0.02738272  0.06774916  0.03729259 -0.02105292  0.07504476]

4. Fashion-CLIP Vector (Đặc trưng đa phương thức - Dense):
   - Số chiều (Shape): (512,)
  